# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook uses the starter dataset, a client-holdout split, and the repo's rule baseline. I compare logistic regression, a shallow decision tree, and a random forest on the same held-out clients, then read the errors before trusting the top score.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I am using a small family of supervised rankers: logistic regression, a shallow decision tree, and a random forest. This is a queue-ranking problem, so the goal is precision at the top of the list, not raw accuracy. Logistic regression is the readable baseline, the tree is a simple nonlinear check, and the random forest is the strongest candidate if the extra flexibility is actually earned.

I am also keeping the feature set honest: client_id and content_id stay out of the model, and the label source columns trend_direction and trend_pct stay out as well. That leaves the observable traffic, freshness, and content-context signals that existed before the outcome window.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 50)

RANDOM_STATE = 42
MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]
FEATURE_COLUMNS = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
LEAKAGE_COLUMNS = {"content_id", "client_id", "trend_direction", "trend_pct"}


def find_raw_path() -> Path:
    candidates = [
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("../../data/raw/content_refresh_anonymized.csv"),
        Path("../../../data/raw/content_refresh_anonymized.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")


RAW_PATH = find_raw_path()
raw = pd.read_csv(RAW_PATH)

for column in [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct",
]:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

for column in [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "trend_direction",
]:
    raw[column] = raw[column].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

for column in [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct",
]:
    raw[column] = raw[column].replace([np.inf, -np.inf], np.nan).fillna(0)

frame = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy().reset_index(drop=True)
frame["is_declining_label"] = frame["trend_direction"].str.lower().eq("down").astype(int)
frame["log_impressions_90d"] = np.log1p(frame["impressions_90d"])
frame["log_clicks_90d"] = np.log1p(frame["clicks_90d"])
frame["log_sessions_90d"] = np.log1p(frame["sessions_90d"])
frame["log_ai_sessions_90d"] = np.log1p(frame["ai_sessions_90d"])
frame["has_clicks"] = (frame["clicks_90d"] > 0).astype(int)
frame["has_ai_sessions"] = (frame["ai_sessions_90d"] > 0).astype(int)
frame["measurable_opportunity"] = ((frame["impressions_90d"] >= 100) & (frame["sessions_90d"] > 0)).astype(int)

print(f"Loaded {len(frame):,} usable rows from {RAW_PATH}")
print(f"Clients: {frame['client_id'].nunique():,}")
print(f"Declining-label rate: {frame['is_declining_label'].mean():.3f}")
print(f"Feature columns used by the models: {len(FEATURE_COLUMNS)}")
print(f"Leakage columns excluded from the model: {sorted(LEAKAGE_COLUMNS)}")
print("First five model features:")
print(FEATURE_COLUMNS[:5])

Loaded 30,000 usable rows from ../../data/raw/content_refresh_anonymized.csv
Clients: 32
Declining-label rate: 0.542
Feature columns used by the models: 26
Leakage columns excluded from the model: ['client_id', 'content_id', 'trend_direction', 'trend_pct']
First five model features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count']


## 2. Split design

I am using a client-holdout split: about 20 percent of clients are held out together, so pages from the same client do not appear in both train and test. That is the honest split for this dataset because content from one client is correlated, and a random row split would leak client-specific patterns into the test set.

The split is still row-level for evaluation, but the grouping is by client. If the client holdout ever breaks class balance, I fall back to a stratified row holdout, but the client split is the first choice.

In [2]:
def make_client_holdout_split(frame: pd.DataFrame, target: pd.Series) -> tuple[np.ndarray, np.ndarray, str]:
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()

    if len(unique_clients) >= 5:
        rng = np.random.default_rng(RANDOM_STATE)
        shuffled_clients = rng.permutation(unique_clients)
        test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
        test_clients = set(shuffled_clients[:test_client_count])
        test_mask = client_series.isin(test_clients).to_numpy()
        train_indices = all_indices[~test_mask]
        test_indices = all_indices[test_mask]

        if (
            len(train_indices) > 0
            and len(test_indices) > 0
            and target.iloc[train_indices].nunique() == 2
            and target.iloc[test_indices].nunique() == 2
        ):
            return train_indices, test_indices, "client_holdout"

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=target,
    )
    return np.asarray(train_indices), np.asarray(test_indices), "stratified_row_holdout"


target = frame["is_declining_label"].astype(int)
train_indices, test_indices, split_strategy = make_client_holdout_split(frame, target)

train_frame = frame.iloc[train_indices].copy().reset_index(drop=True)
test_frame = frame.iloc[test_indices].copy().reset_index(drop=True)
y_train = train_frame["is_declining_label"].astype(int)
y_test = test_frame["is_declining_label"].astype(int)
X_train = train_frame[FEATURE_COLUMNS].copy()
X_test = test_frame[FEATURE_COLUMNS].copy()

print(f"Split strategy: {split_strategy}")
print(f"Train rows: {len(train_frame):,} across {train_frame['client_id'].nunique():,} clients")
print(f"Test rows:  {len(test_frame):,} across {test_frame['client_id'].nunique():,} clients")
print(f"Train declining rate: {y_train.mean():.3f}")
print(f"Test declining rate:  {y_test.mean():.3f}")
print(f"Client overlap between splits: {len(set(train_frame['client_id']).intersection(set(test_frame['client_id'])))}")

Split strategy: client_holdout
Train rows: 27,675 across 26 clients
Test rows:  2,325 across 6 clients
Train declining rate: 0.555
Test declining rate:  0.391
Client overlap between splits: 0


## 3. Train + compare vs my baseline

I fit three models on the same client-holdout split and compare them to the repo's rule baseline on the same held-out rows and the same ranking metric. The headline metric is precision@50 because the queue is used as a short top-K review list, and the table also shows the base rate so the score is never read in isolation.

On this split, the random forest wins: precision@50 is 0.68 versus 0.32 for the rule baseline. That is a real lift, but I still keep the simpler models in the table because the gap between logistic regression, the tree, and the forest tells me how much nonlinear structure the data is actually giving me.

In [3]:
def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def percentile_rank(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)


def normalize(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    minimum = values.min()
    maximum = values.max()
    if not np.isfinite(minimum) or not np.isfinite(maximum) or maximum == minimum:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - minimum) / (maximum - minimum)


def baseline_refresh_score(frame: pd.DataFrame) -> pd.Series:
    visibility_score = percentile_rank(np.log1p(frame["impressions_90d"]))
    freshness_risk_score = percentile_rank(frame["days_since_last_update"])
    position_opportunity_score = (
        (1 - normalize(frame["avg_position"].clip(lower=1, upper=50)))
        * visibility_score
        * (frame["avg_position"] > 0).astype(int)
    )
    depth_gap_score = (1 - percentile_rank(frame["word_count"])) * visibility_score
    return (
        0.40 * visibility_score
        + 0.30 * freshness_risk_score
        + 0.25 * position_opportunity_score
        + 0.05 * depth_gap_score
    ).clip(0, 1)


def precision_at_k(y_true: pd.Series, scores: np.ndarray, k: int) -> float:
    scored = pd.DataFrame({"y": y_true.to_numpy(), "score": np.asarray(scores, dtype=float)})
    top = scored.sort_values("score", ascending=False).head(min(k, len(scored)))
    return float(top["y"].mean()) if len(top) else 0.0


def evaluate_scores(y_true: pd.Series, scores: np.ndarray) -> dict[str, float]:
    predicted = (np.asarray(scores, dtype=float) >= 0.5).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "precision_at_50": float(precision_at_k(y_true, scores, 50)),
        "precision_at_20": float(precision_at_k(y_true, scores, 20)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
    }


preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), MODEL_NUMERIC_FEATURES),
        ("categorical", make_one_hot_encoder(), MODEL_CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

model_specs = {
    "logistic_regression": LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
}

fitted_models: dict[str, Pipeline] = {}
metric_rows: list[dict[str, float | str]] = []
baseline_test_scores = baseline_refresh_score(test_frame)
baseline_metrics = evaluate_scores(y_test, baseline_test_scores)

for model_name, estimator in model_specs.items():
    pipeline = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    fitted_models[model_name] = pipeline
    probability_scores = pipeline.predict_proba(X_test)[:, 1]
    metrics = evaluate_scores(y_test, probability_scores)
    metric_rows.append({"model": model_name, **metrics})

metric_rows.append({"model": "baseline_rules", **baseline_metrics})
metrics_df = pd.DataFrame(metric_rows)
metrics_df = metrics_df[["model", "roc_auc", "average_precision", "precision_at_20", "precision_at_50", "recall", "f1"]]
metrics_df = metrics_df.sort_values(["precision_at_50", "average_precision", "roc_auc"], ascending=False).reset_index(drop=True)

best_model_name = metrics_df.loc[metrics_df["model"] != "baseline_rules", "model"].iloc[0]
best_pipeline = fitted_models[best_model_name]
best_test_scores = best_pipeline.predict_proba(X_test)[:, 1]
best_predictions = (best_test_scores >= 0.5).astype(int)

print(f"Test base rate: {y_test.mean():.3f}")
display(metrics_df)
print(f"Best model by precision@50: {best_model_name}")
print(f"Baseline precision@50: {baseline_metrics['precision_at_50']:.3f}")
print(f"Best model precision@50: {float(metrics_df.loc[metrics_df['model'] == best_model_name, 'precision_at_50'].iloc[0]):.3f}")

Test base rate: 0.391


,model,roc_auc,average_precision,precision_at_20,precision_at_50,recall,f1
0,random_forest,0.747355,0.610081,0.70,0.68,0.741474,0.638258
1,decision_tree,0.741520,0.575319,0.45,0.58,0.716172,0.633885
2,logistic_regression,0.703678,0.524754,0.35,0.40,0.558856,0.564444
3,baseline_rules,0.647799,0.479830,0.25,0.32,0.542354,0.528403


Best model by precision@50: random_forest
Baseline precision@50: 0.320
Best model precision@50: 0.680


## 4. Errors and interpretation

I read the errors from the best model, not just the score. The two questions I care about are: where does it fail, and which features move the ranking enough to matter? A good model should make understandable mistakes, not just produce a bigger number.

Here the random forest mainly misses a small set of low-signal declining pages and occasionally over-calls high-demand pages that look risky because of position and freshness. The permutation-importance check says the model leans most on search exposure, freshness depth, CTR, and engagement signals, which is the kind of story I want to see.

In [4]:
test_results = test_frame[
    [
        "content_type",
        "main_intent",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "sessions_90d",
        "avg_position",
        "ctr",
        "days_with_impressions",
        "days_with_sessions",
        "is_declining_label",
    ]
].copy()
test_results["baseline_score"] = baseline_test_scores
test_results["best_model_probability"] = best_test_scores
test_results["predicted_label"] = best_predictions
test_results["error_type"] = np.select(
    [
        (test_results["is_declining_label"] == 0) & (test_results["predicted_label"] == 1),
        (test_results["is_declining_label"] == 1) & (test_results["predicted_label"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

false_positive_cases = (
    test_results.query("error_type == 'false_positive'")
    .sort_values("best_model_probability", ascending=False)
    .head(3)
    .assign(
        note=lambda data: np.where(
            data["impressions_90d"] >= 500,
            "high-demand page with the wrong decline signal",
            "small page that still looks risky to the model",
        )
    )
)

false_negative_cases = (
    test_results.query("error_type == 'false_negative'")
    .sort_values("best_model_probability", ascending=True)
    .head(3)
    .assign(
        note=lambda data: np.where(
            data["avg_position"].between(1, 10, inclusive="both"),
            "visible page where the model underreacted to the decline pattern",
            "declining page with a weaker signal mix",
        )
    )
)

case_columns = [
    "content_type",
    "main_intent",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "is_declining_label",
    "best_model_probability",
    "baseline_score",
    "note",
]

print(f"Best model on the held-out clients: {best_model_name}")
print("\nThree false positives to inspect:")
display(false_positive_cases[case_columns])
print("\nThree false negatives to inspect:")
display(false_negative_cases[case_columns])

error_summary = (
    test_results.assign(
        false_positive=(test_results["error_type"] == "false_positive").astype(int),
        false_negative=(test_results["error_type"] == "false_negative").astype(int),
    )
    .groupby("content_type", dropna=False)
    .agg(
        n=("is_declining_label", "size"),
        declining_rate=("is_declining_label", "mean"),
        false_positive_rate=("false_positive", "mean"),
        false_negative_rate=("false_negative", "mean"),
        mean_probability=("best_model_probability", "mean"),
    )
    .sort_values("n", ascending=False)
)

position_bins = pd.cut(
    test_results["avg_position"],
    bins=[-0.1, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"],
)
position_error_summary = (
    test_results.assign(
        position_bin=position_bins,
        false_positive=(test_results["error_type"] == "false_positive").astype(int),
        false_negative=(test_results["error_type"] == "false_negative").astype(int),
    )
    .groupby("position_bin", dropna=False)
    .agg(
        n=("is_declining_label", "size"),
        declining_rate=("is_declining_label", "mean"),
        false_positive_rate=("false_positive", "mean"),
        false_negative_rate=("false_negative", "mean"),
    )
)

print("\nError summary by content type:")
display(error_summary)
print("\nError summary by position tier:")
display(position_error_summary)

importance = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_frame = (
    pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "importance": importance.importances_mean,
        "std": importance.importances_std,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nPermutation importance on the best model (scored with average precision):")
display(importance_frame.head(10))
print(
    "Top features are easy to explain if they line up with freshness, demand, and search visibility; "
    "if a top feature looked too perfect or label-like, that would be a leakage warning."
)

Best model on the held-out clients: random_forest

Three false positives to inspect:


,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,sessions_90d,avg_position,ctr,is_declining_label,best_model_probability,baseline_score,note
1752,keyword article,transactional,144,20,5091,30,14.1,0.20,0,0.737431,0.717287,high-demand page with the wrong decline signal
1778,keyword article,informational,125,20,1076,9,25.6,0.09,0,0.735212,0.604215,high-demand page with the wrong decline signal
1791,keyword article,informational,112,20,369,2,21.8,0.00,0,0.733797,0.572784,small page that still looks risky to the model



Three false negatives to inspect:


,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,sessions_90d,avg_position,ctr,is_declining_label,best_model_probability,baseline_score,note
456,keyword article,informational,91,1,1,1,0.0,0.0,1,0.081668,0.035194,declining page with a weaker signal mix
306,feedly article,unknown,308,20,3,1,0.0,0.0,1,0.082714,0.270371,declining page with a weaker signal mix
2094,keyword article,informational,104,8,3,1,0.7,0.0,1,0.151622,0.185572,declining page with a weaker signal mix



Error summary by content type:


...


Error summary by position tier:


,n,declining_rate,false_positive_rate,false_negative_rate
position_bin,,,,
top_3,520,0.115385,0.007692,0.082692
page_1,1062,0.436911,0.252354,0.129944
page_2,402,0.527363,0.343284,0.079602
page_3_5,282,0.485816,0.365248,0.067376
deep,59,0.610169,0.271186,0.050847



Permutation importance on the best model (scored with average precision):


,feature,importance,std
0,days_with_impressions,0.079208,0.007447
1,log_impressions_90d,0.028771,0.004097
2,ctr,0.023523,0.006871
3,scroll_rate,0.009803,0.004703
4,search_volume,0.009634,0.001385
5,log_clicks_90d,0.008346,0.002045
6,avg_position,0.007866,0.005422
7,main_intent,0.002712,0.000140
8,days_with_sessions,0.002591,0.000827
9,engagement_rate,0.002492,0.000850


Top features are easy to explain if they line up with freshness, demand, and search visibility; if a top feature looked too perfect or label-like, that would be a leakage warning.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.